In [ ]:
# Cell 1 — Run this once to create/update the Databricks Job
# Uses the notebook session token which has full API access on CE

import requests
import json

token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get().rstrip("/")

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type":  "application/json",
}

wheel_path = "/Volumes/workspace/default/trade-analytics/wheels/trade_analytics-0.1.0-py3-none-any.whl"
nb_base    = "/Workspace/Shared/trade-analytics-lakehouse/databricks_notebooks"

job_payload = {
    "name": "trade-analytics-pipeline-DEV",
    "environments": [
        {
            "environment_key": "trade_analytics_env",
            "spec": {
                "client": "1",
                "dependencies": [
                    wheel_path,
                ],
            },
        }
    ],
    "tasks": [
        {
            "task_key": "bronze_ingest",
            "description": "Ingest trades.jsonl → Bronze Delta",
            "environment_key": "trade_analytics_env",
            "notebook_task": {
                "notebook_path": f"{nb_base}/01_ingest_bronze",
                "source": "WORKSPACE",
            },
            "timeout_seconds": 1800,
            "max_retries": 1,
        },
        {
            "task_key": "silver_transform",
            "description": "PySpark Bronze → Silver Delta",
            "depends_on": [{"task_key": "bronze_ingest"}],
            "environment_key": "trade_analytics_env",
            "notebook_task": {
                "notebook_path": f"{nb_base}/02_silver_transform",
                "source": "WORKSPACE",
            },
            "timeout_seconds": 3600,
            "max_retries": 1,
        },
        {
            "task_key": "dbt_gold",
            "description": "dbt run + test → Gold Delta marts",
            "depends_on": [{"task_key": "silver_transform"}],
            "environment_key": "trade_analytics_env",
            "notebook_task": {
                "notebook_path": f"{nb_base}/03_run_dbt",
                "source": "WORKSPACE",
            },
            "timeout_seconds": 3600,
            "max_retries": 0,
        },
    ],
}

# Upsert — update if exists, create if not
resp  = requests.get(f"{host}/api/2.1/jobs/list", headers=headers)
jobs  = resp.json().get("jobs", [])
match = [j for j in jobs if j["settings"]["name"] == job_payload["name"]]

if match:
    job_id = match[0]["job_id"]
    resp   = requests.post(
        f"{host}/api/2.1/jobs/reset",
        headers=headers,
        json={"job_id": job_id, "new_settings": job_payload},
    )
    print(f"Job updated (id={job_id}) → {resp.status_code} ✓")
else:
    resp = requests.post(
        f"{host}/api/2.1/jobs/create",
        headers=headers,
        json=job_payload,
    )
    job_id = resp.json().get("job_id")
    print(f"Job created (id={job_id}) → {resp.status_code} ✓")

# Verify
resp = requests.get(f"{host}/api/2.1/jobs/get?job_id={job_id}", headers=headers)
print(f"\nJob name   : {resp.json()['settings']['name']}")
print(f"Job id     : {job_id}")
print(f"Task count : {len(resp.json()['settings']['tasks'])}")
print("\nJob deploy complete ✓")